# CloudSRE v2 — Training Notebook

**Two-Phase Training: SFT Warmup → GRPO Reinforcement Learning**

This notebook trains an LLM agent to diagnose and resolve cascading production incidents in a real microservice environment.

- **Environment**: [CloudSRE HF Space](https://huggingface.co/spaces/DarDrax/CloudSRE-Environment)
- **Model**: Qwen2.5-1.5B-Instruct
- **Phase 1**: SFT on 100 expert demonstrations (teaches command syntax)
- **Phase 2**: GRPO against live environment (teaches strategy)


## Setup

In [ ]:
# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q httpx matplotlib

In [ ]:
# Clone the repository
!git clone https://github.com/Harikishanth/CloudSRE.git
%cd CloudSRE

## Phase 1: SFT Warmup

Fine-tune the base model on 100 expert SRE demonstrations.
This teaches the model valid command formats: `restart_service`, `queue drain`, `cat /var/log/...`

In [ ]:
!python sft_warmup.py \
    --model-id unsloth/Qwen2.5-1.5B-Instruct \
    --epochs 3 \
    --batch-size 2 \
    --lr 2e-4

## Phase 2: GRPO Training (Reinforcement Learning)

Train the SFT checkpoint against the **live** CloudSRE environment using GRPO.
The model interacts with real services, receives dense rewards, and learns SRE strategy.

In [ ]:
ENV_URL = "https://dardrax-cloudsre-environment.hf.space"

!python train_colab.py \
    --env-url {ENV_URL} \
    --model-id ./cloudsre-sft-checkpoint \
    --task-id warmup \
    --episodes 50 \
    --no-hints

## Plot Training Curves

Generate loss and reward plots and save as .png files.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import os

# --- Plot SFT Loss Curve ---
sft_log = "cloudsre-sft-checkpoint/training_log.json"
if os.path.exists(sft_log):
    with open(sft_log) as f:
        sft_data = json.load(f)
    steps = [d['step'] for d in sft_data]
    losses = [d['loss'] for d in sft_data]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps, losses, color='#e74c3c', linewidth=2)
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Phase 1: SFT Loss Curve (100 Expert Demos)', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('sft_loss_curve.png', dpi=150)
    plt.show()
    print('Saved: sft_loss_curve.png')
else:
    print(f'SFT log not found at {sft_log}')
    print('If SFT used HF Trainer, check for checkpoints:')
    !ls cloudsre-sft-checkpoint/ 2>/dev/null || echo 'No checkpoint dir'

In [ ]:
# --- Plot GRPO Reward Curve ---
grpo_log = "training_rewards.json"
if os.path.exists(grpo_log):
    with open(grpo_log) as f:
        rewards_data = json.load(f)
    episodes = list(range(1, len(rewards_data) + 1))
    rewards = [d['reward'] for d in rewards_data]
    resolved = [d.get('resolved', False) for d in rewards_data]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Reward over episodes
    ax1.plot(episodes, rewards, color='#3498db', linewidth=2, alpha=0.7)
    # Rolling average
    window = min(10, len(rewards))
    if window > 1:
        rolling = [sum(rewards[max(0,i-window):i])/min(i, window) for i in range(1, len(rewards)+1)]
        ax1.plot(episodes, rolling, color='#e74c3c', linewidth=3, label=f'{window}-ep rolling avg')
    ax1.set_xlabel('Episode', fontsize=12)
    ax1.set_ylabel('Total Reward', fontsize=12)
    ax1.set_title('Phase 2: GRPO Reward Curve', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Resolution rate over time
    cumulative_rate = [sum(resolved[:i+1])/(i+1)*100 for i in range(len(resolved))]
    ax2.plot(episodes, cumulative_rate, color='#2ecc71', linewidth=2)
    ax2.set_xlabel('Episode', fontsize=12)
    ax2.set_ylabel('Cumulative Resolution Rate (%)', fontsize=12)
    ax2.set_title('Resolution Rate Over Training', fontsize=14)
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('reward_curve.png', dpi=150)
    plt.show()
    print('Saved: reward_curve.png')
else:
    print(f'GRPO reward log not found at {grpo_log}')
    print('The training script should output this file.')

## Evaluation

Run the trained agent against all tiers to get formal metrics.

In [ ]:
!python evaluate.py \
    --env-url {ENV_URL} \
    --model-dir ./cloudsre-agent \
    --episodes-per-tier 10 \
    --output eval_results.json

In [ ]:
# Display evaluation results
if os.path.exists('eval_results.json'):
    with open('eval_results.json') as f:
        results = json.load(f)
    print('=' * 60)
    print('EVALUATION RESULTS')
    print('=' * 60)
    for tier, data in results.items():
        print(f"  {tier:15s}: {data['resolution_rate']:5.1f}% | {data['avg_steps']:.1f} steps | {data['avg_reward']:+.2f} reward")
    print('=' * 60)

## Save Artifacts

Download the training plots and commit them to your repo.

In [ ]:
# Commit plots to repo so they pass validation
!git add sft_loss_curve.png reward_curve.png eval_results.json 2>/dev/null
!git commit -m 'results: training curves and evaluation' 2>/dev/null
!git push origin main 2>/dev/null

# Also download locally
from google.colab import files
for f in ['sft_loss_curve.png', 'reward_curve.png', 'eval_results.json']:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}')